In [1]:
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [3]:
base_dir_path = Path(
    "/eodc/private/tuwgeo/users/mabdelaa/repos/vv_vh_fusion/data/EQUI7_AS020M"
)

imgs = [
    img
    for img in base_dir_path.glob("*/*.tif")
    if "FLOOD-EXPF" in img.name and "VV" in img.name
]

# imgs = imgs[2:]
for img in tqdm(imgs):
    print(img.name)

  0%|          | 0/5 [00:00<?, ?it/s]

FLOOD-EXPF_20210713T121329__VV_A012_E033N024T3_AS020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20210713T121354__VV_A012_E033N024T3_AS020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20210713T121329__VV_A012_E036N024T3_AS020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20210713T121354__VV_A012_E036N024T3_AS020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20210713T121304__VV_A012_E036N024T3_AS020M_V1M3R1_S1_TUWIEN.tif


In [7]:
def get_corresponding_file_paths(flood_vv: Path):
    parent = flood_vv.parent

    # --- SAR-derived products ---
    flood_vh = parent / flood_vv.name.replace("VV", "VH")

    uncert_vv = parent / flood_vv.name.replace("FLOOD", "UNCERT")
    uncert_vh = parent / flood_vv.name.replace("FLOOD", "UNCERT").replace("VV", "VH")

    flood_likelihood_vv = parent / flood_vv.name.replace("FLOOD", "FLOOD_LIKELIHOOD")
    flood_likelihood_vh = parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_LIKELIHOOD"
    ).replace("VV", "VH")

    flood_probability_vv = parent / flood_vv.name.replace("FLOOD", "FLOOD_PROBABILITY")
    flood_probability_vh = parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_PROBABILITY"
    ).replace("VV", "VH")

    # --- Land cover (ESA WorldCover) ---
    lc_candidates = list(parent.glob("ESA-WorldCover-2021-V200*.tif"))

    assert len(lc_candidates) == 1, (
        f"Expected exactly one ESA WorldCover file in {parent}, "
        f"found {len(lc_candidates)}"
    )

    landcover = lc_candidates[0]

    # --- Check existence ---
    for file_path in [
        flood_vv,
        flood_vh,
        uncert_vv,
        uncert_vh,
        flood_likelihood_vv,
        flood_likelihood_vh,
        flood_probability_vv,
        flood_probability_vh,
        landcover,
    ]:
        assert file_path.exists(), f"File {file_path} does not exist."

    return {
        "flood_vv": flood_vv,
        "flood_vh": flood_vh,
        "uncert_vv": uncert_vv,
        "uncert_vh": uncert_vh,
        "flood_likelihood_vv": flood_likelihood_vv,
        "flood_likelihood_vh": flood_likelihood_vh,
        "flood_probability_vv": flood_probability_vv,
        "flood_probability_vh": flood_probability_vh,
        "landcover": landcover,
    }


def read_rasters(paths):
    arrays = {}

    # Read metadata from flood_vv only
    with rasterio.open(paths["flood_vv"]) as src:
        arrays["flood_vv"] = src.read(1)
        meta = src.meta

    for key, path in paths.items():
        if key == "flood_vv":
            continue
        with rasterio.open(path) as src:
            arrays[key] = src.read(1)

    return arrays, meta


def fuse_flood_category(
    flood_vv_data,
    flood_vh_data,
    flood_likelihood_vv_data,
    flood_likelihood_vh_data,
    nodata=np.nan,
):
    """
    Fuse VV/VH flood masks and likelihoods into:
      - fused_flood: uint8 categories {0=no-flood, 1=VV-flood, 2=VH-only-flood}
      - fused_likelihood: same dtype as input likelihood arrays, nodata propagated

    This implementation is defensive:
      - works when arrays are float or integer
      - supports nodata as np.nan or numeric sentinel
      - checks shapes and raises informative errors
    """

    # ---- Basic sanity checks ----
    if not (
        flood_vv_data.shape
        == flood_vh_data.shape
        == flood_likelihood_vv_data.shape
        == flood_likelihood_vh_data.shape
    ):
        raise ValueError("All input arrays must have the same shape.")

    # Convert inputs to numpy arrays (in case they are memory-mapped or similar)
    fv = np.asarray(flood_vv_data)
    fh = np.asarray(flood_vh_data)
    lv = np.asarray(flood_likelihood_vv_data)
    lh = np.asarray(flood_likelihood_vh_data)

    # ---- nodata helper ----
    def mask_is_nodata(arr):
        """Return boolean mask of nodata values for arr given provided nodata sentinel."""
        if np.isnan(nodata):
            # treat NaN as nodata (works for float arrays)
            # For integer arrays, np.isnan will raise, so handle with np.asarray then try/except
            try:
                return np.isnan(arr)
            except TypeError:
                # integer array: there are no NaNs -> return all False
                return np.zeros(arr.shape, dtype=bool)
        else:
            # numeric sentinel: compare (works for float or int)
            return arr == nodata

    # Masks for likelihood nodata
    lv_nodata = mask_is_nodata(lv)
    lh_nodata = mask_is_nodata(lh)

    # Masks for flood input nodata (for final propagation)
    fv_nodata = mask_is_nodata(fv)
    fh_nodata = mask_is_nodata(fh)

    # ---- Category masks ----
    # A pixel is flood in VV if fv == 1 (works for float/int)
    vv_mask = fv == 1
    # VH-only means VH says flood AND VV does not
    vh_only_mask = (fh == 1) & (~vv_mask)
    # Non-flood: neither says flood
    non_flood_mask = ~(vv_mask | vh_only_mask)

    # ---- fused_flood (uint8) ----
    fused_flood = np.zeros_like(fv, dtype=np.uint8)
    fused_flood[vv_mask] = 1
    fused_flood[vh_only_mask] = 2
    # Default 0 is already no-flood. If you want a different nodata code for fused_flood,
    # change dtype and fill value here.

    # ---- fused_likelihood (same dtype as inputs) ----
    # Use lv.dtype if that is numeric; else cast to float32 for safety
    out_dtype = lv.dtype if np.issubdtype(lv.dtype, np.number) else np.float32
    fused_likelihood = np.full(lv.shape, nodata, dtype=out_dtype)

    # Assign flood pixel likelihoods directly
    # but be careful: if source likelihood is nodata, keep nodata
    valid_vv_for_flood = vv_mask & (~lv_nodata)
    valid_vv_but_nodata = vv_mask & lv_nodata
    fused_likelihood[valid_vv_for_flood] = lv[valid_vv_for_flood]
    # leave fused_likelihood as nodata where lv_nodata is True

    valid_vh_for_flood = vh_only_mask & (~lh_nodata)
    fused_likelihood[valid_vh_for_flood] = lh[valid_vh_for_flood]

    # ---- Non-flood fusion with nodata handling ----
    both_valid = non_flood_mask & (~lv_nodata) & (~lh_nodata)
    fused_likelihood[both_valid] = (lv[both_valid] + lh[both_valid]) / 2.0

    only_vv_valid = non_flood_mask & (~lv_nodata) & (lh_nodata)
    fused_likelihood[only_vv_valid] = lv[only_vv_valid]

    only_vh_valid = non_flood_mask & (lv_nodata) & (~lh_nodata)
    fused_likelihood[only_vh_valid] = lh[only_vh_valid]

    # If both likelihoods missing for non-flood, fused stays nodata

    # ---- Final nodata propagation from flood inputs ----
    flood_input_nodata = fv_nodata | fh_nodata
    if np.any(flood_input_nodata):
        fused_likelihood[flood_input_nodata] = nodata
        # Optionally set fused_flood to a nodata code if you want:
        # fused_flood[flood_input_nodata] = 255

    return fused_flood, fused_likelihood

In [8]:
for img_path in tqdm(imgs):
    paths = get_corresponding_file_paths(img_path)
    arrays, meta = read_rasters(paths)
    fused_flood, fused_likelihood = fuse_flood_category(
        arrays["flood_vv"],
        arrays["flood_vh"],
        arrays["flood_likelihood_vv"],
        arrays["flood_likelihood_vh"],
    )
    save_path = base_dir_path / "Fusion/Catagory_Based"
    save_path.mkdir(parents=True, exist_ok=True)
    fused_flood_path = save_path / img_path.name.replace(
        "FLOOD-EXPF", "FUSED-FLOOD-EXPF"
    ).replace("VV", "FUSED")
    fused_likelihood_path = save_path / img_path.name.replace(
        "FLOOD-EXPF", "FUSED-FLOOD-LIKELIHOOD"
    ).replace("VV", "FUSED")
    with rasterio.open(fused_flood_path, "w", **meta) as dst:
        dst.write(fused_flood, 1)
    with rasterio.open(fused_likelihood_path, "w", **meta) as dst:
        dst.write(fused_likelihood, 1)

  0%|          | 0/111 [00:00<?, ?it/s]

AssertionError: File /eodc/private/tuwgeo/users/mabdelaa/data/s1floodat/italy/2023_5/FLOOD/V1M3R1/EQUI7_EU020M/E048N012T3/FLOOD-EXPF_20230513T165052__VH_A146_E048N012T3_EU020M_V1M3R1_S1_TUWIEN.tif does not exist.